In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import geopandas as gpd


In [ ]:
with open('network_graph.pickle', 'rb') as f:
    G1 = pickle.load(f)

In [ ]:
data = []
for node, attrs in G1.nodes(data=True):
    if "county" in attrs and "state" in attrs:
        row = {
            "state": attrs["state"],
            "county": attrs["county"],
            "opinion": attrs.get("baseline_opinion")
        }
        data.append(row)

df = pd.DataFrame(data)

In [ ]:
df.head()

In [ ]:
county_counts = df[['county', 'state']].value_counts().reset_index()

In [ ]:
county_counts.columns

In [ ]:
county_counts["name"] = county_counts["county"] + ", " + county_counts["state"]

In [ ]:
county_counts.head()

In [ ]:
import matplotlib.pyplot as plt
top_n = 20
plot_data = county_counts.head(top_n)

plt.figure()
plt.barh(plot_data['name'], plot_data['count'])
plt.xlabel('Count')
plt.ylabel('County, State')
plt.title(f'Top {top_n} County Counts')
plt.gca().invert_yaxis()

plt.show()

In [ ]:
counties = gpd.read_file("tl_2024_us_county/tl_2024_us_county.shp")

In [ ]:
counties.columns

In [ ]:
counties.head()

In [ ]:
county_counts['county'] = county_counts['county'].str.upper().str.strip()
county_counts['state'] = county_counts['state'].str.upper().str.strip()

In [ ]:
counties['county'] = counties['NAME'].str.upper().str.replace(' COUNTY', '').str.strip()

In [ ]:
state_fips = {
    "01": "ALABAMA",
    "02": "ALASKA",
    "04": "ARIZONA",
    "05": "ARKANSAS",
    "06": "CALIFORNIA",
    "08": "COLORADO",
    "09": "CONNECTICUT",
    "10": "DELAWARE",
    "12": "FLORIDA",
    "13": "GEORGIA",
    "15": "HAWAII",
    "16": "IDAHO",
    "17": "ILLINOIS",
    "18": "INDIANA",
    "19": "IOWA",
    "20": "KANSAS",
    "21": "KENTUCKY",
    "22": "LOUISIANA",
    "23": "MAINE",
    "24": "MARYLAND",
    "25": "MASSACHUSETTS",
    "26": "MICHIGAN",
    "27": "MINNESOTA",
    "28": "MISSISSIPPI",
    "29": "MISSOURI",
    "30": "MONTANA",
    "31": "NEBRASKA",
    "32": "NEVADA",
    "33": "NEW HAMPSHIRE",
    "34": "NEW JERSEY",
    "35": "NEW MEXICO",
    "36": "NEW YORK",
    "37": "NORTH CAROLINA",
    "38": "NORTH DAKOTA",
    "39": "OHIO",
    "40": "OKLAHOMA",
    "41": "OREGON",
    "42": "PENNSYLVANIA",
    "44": "RHODE ISLAND",
    "45": "SOUTH CAROLINA",
    "46": "SOUTH DAKOTA",
    "47": "TENNESSEE",
    "48": "TEXAS",
    "49": "UTAH",
    "50": "VERMONT",
    "51": "VIRGINIA",
    "53": "WASHINGTON",
    "54": "WEST VIRGINIA",
    "55": "WISCONSIN",
    "56": "WYOMING"
}

counties['state'] = counties['STATEFP'].map(state_fips)
counties['state'] = counties['state'].str.upper()

In [ ]:
counties.head()

In [ ]:
county_counts['county'] = county_counts['county'].str.upper().str.replace(' COUNTY', '').str.strip()

In [ ]:
county_counts.head()

In [ ]:
counties[counties["state"] == "ALASKA"]

In [ ]:
merged = counties.merge(
    county_counts,
    on=['county', 'state'],
    how='left'
)

In [ ]:
merged[merged["state"] == "ALASKA"]

In [ ]:
county_counts[(county_counts["state"] == "NEBRASKA")]

In [ ]:
merged.loc[merged['count'].isna(), 'count'] = 0


In [ ]:
merged.shape

In [ ]:
merged['count'].notna().mean()

In [ ]:
counties["state"]

In [ ]:
county_counts["state"]

In [ ]:
merged.loc[merged['count'] > 0, 'present'] = True
merged.loc[merged['count'] == 0, 'present'] = False

In [ ]:
merged['present_int'] = merged['present'].astype(int)

state_summary = merged.groupby('state')['present_int'].mean().sort_values()

plt.figure(figsize=(12,6))
plt.bar(state_summary.index, state_summary.values)
plt.xticks(rotation=90)
plt.ylabel("Proportion of counties present")
plt.title("Presence rate by state")
plt.tight_layout()
plt.show()

In [ ]:
state_gdf = merged.dissolve(
    by='state',
    aggfunc={'present_int': 'mean'}
)

In [ ]:
from matplotlib.colors import ListedColormap
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


In [ ]:
gdf = merged

conus = gdf[~gdf['state'].isin(["ALASKA", "HAWAII"] + ["PUETRO RICO"])]
alaska = gdf[gdf['state'] == "ALASKA"]
hawaii = gdf[gdf['state'] == "HAWAII"]

# --- create figure ---
fig = plt.figure(figsize=(14, 9))
ax_main = fig.add_subplot(111)

cmap = ListedColormap(['red', 'green'])
data = [[True, False], [False, True]]

# --- main map (CONUS) ---
conus.plot(
    column='present',
    ax=ax_main,
    cmap=cmap,
    legend=True,
    missing_kwds={"color": "lightgrey"}
)
ax_main.set_xlim(-130, -65) 
ax_main.set_ylim(24, 50)
ax_main.set_title("County Presence Map (USA)")
ax_main.axis('off')

ax_alaska = inset_axes(ax_main, width="25%", height="25%", loc='lower left', borderpad=2)
print(alaska.empty == True)
alaska.plot(
    column='present',
    ax=ax_alaska,
    cmap=cmap,
    legend=False,
    missing_kwds={"color": "lightgrey"}
)
ax_alaska.set_xlim(-250, -130) 
ax_alaska.set_ylim(50, 72)

# ax_alaska.set_title("Alaska", fontsize=8)
ax_alaska.axis('off')

# --- Hawaii inset ---
ax_hawaii = inset_axes(ax_main, width="20%", height="20%", loc='lower right', borderpad=2)

hawaii.plot(
    column='present',
    ax=ax_hawaii,
    cmap=cmap,
    legend=False,
    missing_kwds={"color": "lightgrey"}
)

ax_hawaii.set_xlim(-161, -154)
ax_hawaii.set_ylim(18, 23)
ax_hawaii.set_aspect('equal')
# ax_hawaii.set_title("Hawaii", fontsize=8)
ax_hawaii.axis('off')

plt.tight_layout()
plt.savefig("map.pdf")
plt.show()

In [ ]:
state_gdf = merged.dissolve(
    by='state',
    aggfunc={'present_int': 'mean'}
)

In [ ]:
gdf = merged

cmap = ListedColormap(["#f2f2f2", "#b30000"])

conus = gdf[~gdf['state'].isin(["ALASKA", "HAWAII"] + ["PUErTO RICO"])]
alaska = gdf[gdf['state'] == "ALASKA"]
hawaii = gdf[gdf['state'] == "HAWAII"]

# --- create figure ---
fig = plt.figure(figsize=(14, 9))
ax_main = fig.add_subplot(111)

data = [[True, False], [False, True]]

# --- main map (CONUS) ---
conus.plot(
    column='present_int',
    ax=ax_main,
    cmap=cmap,
    legend=True,
    missing_kwds={"color": "lightgrey"}
)
ax_main.set_xlim(-130, -65) 
ax_main.set_ylim(24, 50)
ax_main.set_title("Presence Rate by State (Heatmap)")
ax_main.axis('off')

# # --- Alaska inset ---
ax_alaska = inset_axes(ax_main, width="25%", height="25%", loc='lower left', borderpad=2)
print(alaska.empty == True)
alaska.plot(
    column='present_int',
    ax=ax_alaska,
    cmap=cmap,
    legend=False,
    missing_kwds={"color": "lightgrey"}
)
ax_alaska.set_xlim(-250, -130) 
ax_alaska.set_ylim(50, 72)


# ax_alaska.set_title("Alaska", fontsize=8)
ax_alaska.axis('off')

# --- Hawaii inset ---
ax_hawaii = inset_axes(ax_main, width="20%", height="20%", loc='lower right', borderpad=2)

hawaii.plot(
    column='present_int',
    ax=ax_hawaii,
    cmap=cmap,
    legend=False,
    missing_kwds={"color": "lightgrey"}
)

ax_hawaii.set_xlim(-161, -154)
ax_hawaii.set_ylim(18, 23)
ax_hawaii.set_aspect('equal')
# ax_hawaii.set_title("Hawaii", fontsize=8)
ax_hawaii.axis('off')

plt.tight_layout()
plt.savefig("map_state.pdf")
plt.show()